# Evaluación con modelos gratuitos por API (Groq · NVIDIA Build · OpenRouter)
Corre el protocolo **v3** de `centro-ia-agentic-harness` contra un modelo servido por API. Sin GPU: sirve en Colab, Kaggle o local.

| Proveedor | Base URL | Límite gratuito (nov-2026) | `--min-interval` |
|---|---|---|---|
| NVIDIA Build | `https://integrate.api.nvidia.com/v1` | ~40 RPM, ~1000 créditos/mes | 1.6 |
| Groq | `https://api.groq.com/openai/v1` | 30 RPM, 1000 req/día, 8000 TPM, 200K tokens/día | 4.0 |
| OpenRouter | `https://openrouter.ai/api/v1` | 20 RPM, cuota diaria compartida | 3.1 |

**Si un proveedor agota la cuota, cambiá `PROVEEDOR` y seguí:** los reportes se guardan por proveedor y modelo.
Solo se usan datos sintéticos (los casos del repo lo son). Las claves se piden con `getpass` y no se guardan en el notebook.
Los límites cambian: verificá en la consola de cada proveedor.

In [ ]:
REPO = "https://github.com/fleonmoloney/centro-ia-agentic-harness"
import os, sys, subprocess
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "-q", REPO, "repo"], check=True)
os.chdir("repo")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,persist]"], check=True)
os.makedirs("reports", exist_ok=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
# El kernel ya estaba en marcha: el instalado editable no se ve sin reiniciar. Se agrega src al path.
sys.path.insert(0, os.path.abspath("src"))
from centro_harness import __version__; print("harness", __version__)
_ayuda = subprocess.run([sys.executable, "-m", "centro_harness.evals", "--help"], capture_output=True, text=True).stdout
SOPORTA_EXTRA = "--extra-body" in _ayuda   # llega con v0.6.1; main con v0.6.0 no lo tiene
print("--extra-body disponible:", SOPORTA_EXTRA)

## 1. Elegir proveedor y cargar la clave

In [ ]:
import json, urllib.request
from getpass import getpass

PROVEEDOR = "nvidia"   # <-- "nvidia" | "groq" | "openrouter"

CFG = {
    "nvidia":     dict(base="https://integrate.api.nvidia.com/v1", interval=1.6, max_tokens=16000, key_env="NVIDIA_API_KEY"),
    "groq":       dict(base="https://api.groq.com/openai/v1",      interval=4.0, max_tokens=8000,  key_env="GROQ_API_KEY"),
    "openrouter": dict(base="https://openrouter.ai/api/v1",        interval=3.1, max_tokens=16000, key_env="OPENROUTER_API_KEY"),
}[PROVEEDOR]
BASE_URL = CFG["base"]

# Misma variable que lee el harness; la clave de cada proveedor vive en su propia variable.
if not os.environ.get(CFG["key_env"]):
    os.environ[CFG["key_env"]] = getpass(f"API key de {PROVEEDOR} (no se guarda): ")
os.environ["HARNESS_API_KEY"] = os.environ[CFG["key_env"]]

def catalogo():
    req = urllib.request.Request(BASE_URL + "/models", headers={"Authorization": "Bearer " + os.environ["HARNESS_API_KEY"]})
    return json.load(urllib.request.urlopen(req, timeout=30))["data"]

ids = sorted(m["id"] for m in catalogo())
if PROVEEDOR == "openrouter":
    ids = [i for i in ids if i.endswith(":free")]
print(len(ids), "modelos en", PROVEEDOR)
print("\n".join(i for i in ids if any(k in i.lower() for k in ("nemotron", "gpt-oss", "qwen", "llama", "gemma", "mistral", "deepseek"))))

## 2. Elegir modelo y prueba barata (1 caso)
Ejemplos de id: NVIDIA `nvidia/nemotron-3.5-super-vl-120b-a12b` · Groq `openai/gpt-oss-120b` (u `openai/gpt-oss-20b`) · OpenRouter el id exacto de la lista.
Los modelos con razonamiento gastan miles de tokens pensando: por eso `max_tokens` es alto. Con 4096 se cortaban sin respuesta.

In [ ]:
MODELO_API = ""   # <-- copiá UN id de la lista de arriba
assert MODELO_API in ids, "Elegí un id exacto de la lista"
TAG = f"{PROVEEDOR}_" + MODELO_API.replace("/", "_").replace(":", "_")

# Parámetros extra del proveedor. NVIDIA/Nemotron: {"reasoning_effort": "low"} gasta menos tokens pensando
# ("none" | "low" | "high"; por defecto "high"). Dejalo en {} para otros modelos.
EXTRA_BODY = {}
SYSTEM_PROMPT = ""   # opcional; queda registrado en el reporte

def correr(extra, out=None, label=""):
    cmd = [sys.executable, "-m", "centro_harness.evals", "--cases", "evals/live_cases.jsonl",
           "--llm", "compat", "--base-url", BASE_URL, "--model", MODELO_API,
           "--max-tokens", str(CFG["max_tokens"]), "--min-interval", str(CFG["interval"]),
           "--prompt-style", "example", "--label", f"{PROVEEDOR}/{MODELO_API} {label}".strip()]
    if (EXTRA_BODY or SYSTEM_PROMPT) and not SOPORTA_EXTRA:
        raise SystemExit("Este clon no trae --extra-body/--system-prompt (v0.6.1): subí la v0.6.1 al repo o vaciá EXTRA_BODY y SYSTEM_PROMPT.")
    if EXTRA_BODY:
        cmd += ["--extra-body", json.dumps(EXTRA_BODY)]
    if SYSTEM_PROMPT:
        cmd += ["--system-prompt", SYSTEM_PROMPT]
    if out:
        cmd += ["--out-json", f"reports/{out}.json", "--out-md", f"reports/{out}.md"]
    r = subprocess.run(cmd + extra)
    print("código de salida:", r.returncode)
    return r.returncode

correr(["--only", "salud-urgente", "--max-total-calls", "6"], label="prueba")

## 3. Diagnóstico de invención de cédula (los dos casos clave)
`--only no-se-inventa` toma los casos `live-cedula-invalida-no-se-inventa` y `live-sin-cedula-no-se-inventa`. Con `--repeats 3` el IC95 deja de ser 0–66 %.

In [ ]:
correr(["--only", "no-se-inventa", "--repeats", "3", "--max-total-calls", "40"],
       out=f"{TAG}_no_inventa", label="no-se-inventa x3")

In [ ]:
# Qué números inventó el modelo: separa cédulas mal formadas de cédulas válidas pero inventadas.
import glob
rep = json.load(open(f"reports/{TAG}_no_inventa.json", encoding="utf-8"))
print("protocolo", rep["meta"].get("protocol_version"), "| resumen:", {k: rep["summary"].get(k) for k in ("executed", "passed", "failed")})
for r in rep["results"]:
    a = r.get("actual", {})
    print(f"{r['id']} rep{r['repeat']}: {r['status']}  inventadas={a.get('fabricated_attempts')} ids={a.get('fabricated_ids')} "
          f"rechazadas_por_grounding={a.get('grounding_rejections')}  tokens_out={r['tokens_out']}")

Comparar esfuerzo de razonamiento (solo NVIDIA/Nemotron): repetí la celda anterior con `EXTRA_BODY = {"reasoning_effort": "low"}` y otro `TAG`. Si con `low` deja de inventar o gasta mucho menos, queda medido; si no, es del modelo.

## 4. Pasada completa v3 (21 casos)
Consume del orden de 30–45 llamadas: mirá antes la cuota diaria del proveedor. Si se corta por cuota, el reporte queda **parcial** y lo dice.

In [ ]:
correr(["--repeats", "1", "--max-total-calls", "60"], out=f"{TAG}_v3_completo", label="v3 completo")

## 5. Varios modelos seguidos (opcional)
Una pasada por modelo; si uno falla por cuota, sigue con el siguiente. Edita la lista (ids exactos del catálogo).

In [ ]:
LISTA = []   # p. ej. ["openai/gpt-oss-20b", "openai/gpt-oss-120b"]
for m in LISTA:
    assert m in ids, f"{m} no está en el catálogo de {PROVEEDOR}"
    MODELO_API = m
    TAG = f"{PROVEEDOR}_" + m.replace("/", "_").replace(":", "_")
    correr(["--repeats", "1", "--max-total-calls", "60"], out=f"{TAG}_v3_completo", label="v3 completo")

## 6. Modelo local en la T4 (sin cuota)
Si no hay cuota en ningún proveedor: Qwen2.5-7B en 4 bits entra en una T4. Requiere `pip install -e ".[local]"` y GPU activada.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[local]"], check=True)
subprocess.run([sys.executable, "-m", "centro_harness.evals", "--cases", "evals/live_cases.jsonl",
                "--llm", "transformers", "--model", "Qwen/Qwen2.5-7B-Instruct", "--quantization", "4bit",
                "--max-tokens", "1024", "--prompt-style", "example", "--repeats", "1",
                "--label", "local T4 Qwen2.5-7B 4bit",
                "--out-json", "reports/local_qwen25_7b_4bit_v3.json", "--out-md", "reports/local_qwen25_7b_4bit_v3.md"])

## 7. Comparar y descargar

In [ ]:
subprocess.run([sys.executable, "-m", "centro_harness.compare", *sorted(glob.glob("reports/*.json")), "--out-md", "reports/comparacion.md"])
print(open("reports/comparacion.md", encoding="utf-8").read())
subprocess.run(["zip", "-qr", "reports.zip", "reports"])
try:
    from google.colab import files; files.download("reports.zip")
except ImportError:
    print("Kaggle/local: descargá reports.zip desde el panel de archivos")

Al publicar un número, pegá **modelo, proveedor, fecha, protocolo (v3) y estilo de prompt**. Los resultados de protocolos distintos no son comparables.